# vector normalisation — procedural drill

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `vector-normalisation`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five vector-normalization patterns that ramp from L2 norm → unit vector → axis-aware keepdim → batch normalize → safe normalize with epsilon floor. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `vector-normalisation`**, which bridges to the bank subtopic `Numpy: Applied patterns and advanced` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "vector-normalisation"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Vector normalisation — quick refresher

**L2 norm:** `||v|| = sqrt(sum_i v_i^2)`. In PyTorch:
- `v.norm()` / `torch.linalg.norm(v)` — for any tensor.
- `v.pow(2).sum().sqrt()` — same thing, explicit.

**Per-row normalize:** `x / x.pow(2).sum(dim=1, keepdim=True).sqrt()`.
The `keepdim=True` is critical — it keeps the reduced axis as size 1 so the divisor broadcasts back.

**Safe normalize:** wrap the divisor in `torch.clamp(norm, min=eps)` to avoid NaN on zero-norm rows.

### Exercise 1 — L2 norm of a 1-D vector

> ```yaml
> Difficulty: ⚪⚪⚪⚪⚪
> Bloom level: Remember
> LO: Recall how to compute the Euclidean norm of a 1-D tensor.
> Keywords: l2-norm, scalar-output, euclidean
> ```

**KCs targeted:** `l2-norm-compute`

Implement `ex1_l2_norm(v)` to compute the Euclidean (L2) norm of a 1-D tensor: `sqrt(sum(v_i^2))`. Output is a 0-D scalar tensor.

Use `torch.linalg.norm`, `torch.norm`, or `v.pow(2).sum().sqrt()` — any of the three is fine.

In [ ]:
def ex1_l2_norm(v: Tensor) -> Tensor:
    """L2 norm of a 1-D tensor. Returns a 0-D scalar."""
    raise NotImplementedError()


def _test_ex1():
    v = t.tensor([3.0, 4.0])                             # known norm = 5
    n = ex1_l2_norm(v)
    assert n.dim() == 0, f'expected scalar (0-D), got shape {tuple(n.shape)}'
    assert t.allclose(n, t.tensor(5.0)), f'expected 5.0, got {n.item()}'

    v2 = t.tensor([1.0, 0.0, 0.0])                   # unit-x
    assert t.allclose(ex1_l2_norm(v2), t.tensor(1.0)), 'unit-x norm should be 1'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_l2_norm(v: Tensor) -> Tensor:
    return v.pow(2).sum().sqrt()
```

**Three equivalent calls:** `v.norm()`, `torch.linalg.norm(v)`, `v.pow(2).sum().sqrt()`. The first two are concise; the third makes the formula explicit and is what `torch.norm` does under the hood.
</details>

### Exercise 2 — normalize a 1-D vector to unit length

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Bloom level: Apply
> LO: Apply scalar division by the norm to produce a unit vector from a 1-D tensor.
> Keywords: unit-vector, scalar-division, normalization
> ```

**KCs targeted:** `unit-vector-divide`

Implement `ex2_unit_vector(v)` to return `v / ||v||`. Output shape matches the input shape, and the result must have norm 1.

Don't use `torch.nn.functional.normalize` — write the division explicitly. The point is to see the shape arithmetic (scalar division).

In [ ]:
def ex2_unit_vector(v: Tensor) -> Tensor:
    """Return v / ||v||. Output shape matches v; result has norm 1."""
    raise NotImplementedError()


def _test_ex2():
    v = t.tensor([3.0, 4.0])
    u = ex2_unit_vector(v)
    assert u.shape == v.shape, f'shape mismatch: {u.shape} vs {v.shape}'
    assert t.allclose(u, t.tensor([0.6, 0.8])), f'expected [0.6, 0.8], got {u.tolist()}'
    assert t.allclose(u.norm(), t.tensor(1.0)), 'unit-vector norm should be 1.0'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_unit_vector(v: Tensor) -> Tensor:
    return v / v.pow(2).sum().sqrt()
```
</details>

### Exercise 3 — norm along an axis with keepdim

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `dim=` and `keepdim=True` to compute per-row norms shaped for broadcast.
> Keywords: axis-aware, keepdim, broadcast-prep
> ```

**KCs targeted:** `norm-along-axis-keepdim`

Implement `ex3_row_norms_keepdim(x)` to compute the L2 norm of every row of a 2-D tensor, **keeping** the reduced axis as size 1.

Input shape: `(N, D)`. Output shape: `(N, 1)` — NOT `(N,)`. The size-1 axis lets the result broadcast back against the input.

Use `keepdim=True` (or `(...,)` axis bookkeeping by hand).

In [ ]:
def ex3_row_norms_keepdim(x: Tensor) -> Tensor:
    """Per-row L2 norm with keepdim. (N, D) → (N, 1)."""
    raise NotImplementedError()


def _test_ex3():
    x = t.tensor([[3.0, 4.0], [0.0, 5.0], [1.0, 2.0]])  # norms: 5, 5, sqrt(5)
    n = ex3_row_norms_keepdim(x)
    assert n.shape == (3, 1), f'expected (3,1), got {tuple(n.shape)}'
    expected = t.tensor([[5.0], [5.0], [(5.0)**0.5]])
    assert t.allclose(n, expected), f'value mismatch: {n.tolist()}'
    # Critical broadcast check: (3, D) / (3, 1) must work without error.
    broadcast_ok = x / n
    assert broadcast_ok.shape == x.shape, 'keepdim should let result broadcast against input'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_row_norms_keepdim(x: Tensor) -> Tensor:
    return x.pow(2).sum(dim=1, keepdim=True).sqrt()
```

**Why `keepdim=True`?** Without it the result is shape `(N,)`, which does NOT broadcast back against `(N, D)` for division — PyTorch would right-align `(N,)` against `(N, D)`'s last dim and complain. With `keepdim=True` you get `(N, 1)` which broadcasts cleanly.
</details>

### Exercise 4 — normalize every row of a batch

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply per-row normalization to a batch of vectors using axis-keepdim broadcast.
> Keywords: batch-normalize, surface-normals, ray-tracing
> ```

**KCs targeted:** `per-row-batch-normalize`

Implement `ex4_batch_unit_vectors(x)` to L2-normalize every row of a batch.

Input shape: `(N, D)`. Output shape: `(N, D)`. Every row of the output should be a unit vector.

This is the canonical operation for converting a batch of raw surface normals into the unit normals needed for shading. Equivalent to `torch.nn.functional.normalize(x, dim=1)` — write it explicitly using the keepdim norm from Exercise 3.

In [ ]:
def ex4_batch_unit_vectors(x: Tensor) -> Tensor:
    """Per-row L2-normalize. (N, D) → (N, D), every row has norm 1."""
    raise NotImplementedError()


def _test_ex4():
    import torch.nn.functional as F
    x = t.tensor([[3.0, 4.0], [0.0, 5.0], [1.0, 2.0], [-1.0, -1.0]])
    u = ex4_batch_unit_vectors(x)
    assert u.shape == x.shape, f'shape mismatch: {u.shape} vs {x.shape}'
    row_norms = u.pow(2).sum(dim=1).sqrt()
    assert t.allclose(row_norms, t.ones(4), atol=1e-6), f'rows should have norm 1, got {row_norms.tolist()}'
    assert t.allclose(u, F.normalize(x, dim=1), atol=1e-6), 'should match F.normalize(x, dim=1)'
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
def ex4_batch_unit_vectors(x: Tensor) -> Tensor:
    return x / x.pow(2).sum(dim=1, keepdim=True).sqrt()
```
</details>

### Exercise 5 — safe normalize with epsilon (zero-norm guard)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize axis-keepdim norm + per-row divide + epsilon floor to normalize safely even when some rows are zero vectors.
> Keywords: numerical-stability, epsilon, nan-guard, multi-kc
> ```

**KCs targeted:** `norm-along-axis-keepdim`, `per-row-batch-normalize`, `safe-zero-norm-normalize`

Implement `ex5_safe_normalize(x, eps=1e-12)` to L2-normalize every row of `x` **without producing NaN** for zero-norm rows.

Input shape: `(N, D)`. Output shape: `(N, D)`. Use `eps` as the floor for the divisor: `divisor = max(norm, eps)`. Zero-norm rows then come out as zero rows (not NaN, not inf).

Strategy: `torch.clamp(norm, min=eps)` is the clean way. Don't add `eps` inside the sqrt — that quietly inflates non-zero norms.

> ⚠️ **Integrative exercise.** This combines 3+ KCs in one expression; empirical work (Lohr et al. ITiCSE 2025) shows 3-concept LLM-generated exercises drop from ~94% to ~40% solvability. Expect a step in difficulty here vs Exercises 1-4.

In [ ]:
def ex5_safe_normalize(x: Tensor, eps: float = 1e-12) -> Tensor:
    """Per-row L2 normalize with eps floor on the divisor.

    Zero-norm rows return as zero rows (no NaN).
    """
    raise NotImplementedError()


def _test_ex5():
    x = t.tensor([
        [3.0, 4.0],     # norm 5
        [0.0, 0.0],     # zero vector — must NOT produce NaN
        [1.0, 0.0],     # already unit
        [-2.0, 0.0],    # norm 2
    ])
    u = ex5_safe_normalize(x)
    assert u.shape == x.shape, f'shape mismatch: {u.shape}'
    assert not t.isnan(u).any(), 'output contains NaN — eps floor not applied'
    assert not t.isinf(u).any(), 'output contains inf — eps floor not applied'
    # Non-zero rows must be properly normalized.
    assert t.allclose(u[0], t.tensor([0.6, 0.8]), atol=1e-6), 'row 0 should be [0.6, 0.8]'
    assert t.allclose(u[1], t.zeros(2)), 'zero-input row should be zero-output row'
    assert t.allclose(u[2], t.tensor([1.0, 0.0])), 'unit-x row should stay unit-x'
    assert t.allclose(u[3], t.tensor([-1.0, 0.0])), 'row 3 should be [-1, 0]'
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_safe_normalize(x: Tensor, eps: float = 1e-12) -> Tensor:
    norm = x.pow(2).sum(dim=1, keepdim=True).sqrt()
    return x / t.clamp(norm, min=eps)
```

**Why `clamp` and not `norm + eps`?**
- `norm + eps`: the divisor is always at least `eps`, so a vector of norm 1.0 gets divided by `1.0 + 1e-12` — a tiny but real bias.
- `clamp(norm, min=eps)`: the divisor is `norm` for any norm ≥ eps and `eps` only for zero-norm rows. No bias for well-conditioned vectors.

**Why eps=1e-12 in float32?** float32 machine epsilon is ~1.19e-7. Any non-zero norm we care about is many orders of magnitude above 1e-12, so `clamp` only kicks in for genuine zeros. Lower than ~1e-20 risks the divisor itself underflowing.
</details>

## Done

Run the cell below to report your progress to Delta Drills. The beacon fires only if all 5 exercises passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1', 'ex2', 'ex3', 'ex4', 'ex5'}

def _dd_feedback_level(num_passed: int) -> str:
    """Map exercise-pass count → arena-rating feedback enum."""
    if num_passed == 5: return 'not_much'
    if num_passed >= 3: return 'somewhat'
    return 'a_lot'

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {len(missing)} exercises still failing: {sorted(missing)}.")
        print("[Delta Drills] not reporting until all 5 pass.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}',
        'subtopics': [DD_SUBTOPIC],
        'feedback': _dd_feedback_level(len(_dd_passed)),
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()